In [1]:
%%writefile /kaggle/working/recommender.py

import gc
import json
from collections import Counter, defaultdict
from pathlib import Path

import catboost as cb
import faiss
import numpy as np
import pandas as pd

from recproj2_candidates import generate_candidates, load_model_and_build_index


EVENT_TYPES = ('clicks', 'carts', 'orders')
TYPE_WEIGHTS = {'clicks': 1.0, 'carts': 3.0, 'orders': 6.0}
CANDIDATE_K = 150
LAST_N = 10
TOP_K = 20


def compute_features_for_session(session_aids, session_types, session_ts,
                                 candidates, event_type, model, *,
                                 prediction_ts, candidate_metadata, popular_items,
                                 candidate_rank_map=None):
    """Признаки пары «сессия — товар» по известной истории; время задано в мс."""
    session_len = len(session_aids)
    unique_items = len(set(session_aids))
    history_counts = Counter()
    first_pos = {}
    last_pos = {}
    item_action_counts = defaultdict(Counter)
    item_last_action = {}
    item_last_event_ts = {}
    item_last_click_ts = {}
    item_last_cart_ts = {}
    item_last_cart_pos = {}
    item_last_order_pos = {}
    events_last_hour = 0
    events_last_day = 0

    # За один проход считаются действия, позиции и давность событий.
    hour_start = prediction_ts - 60 * 60 * 1000
    day_start = prediction_ts - 24 * 60 * 60 * 1000
    for position, (aid, action, ts) in enumerate(
        zip(session_aids, session_types, session_ts)
    ):
        history_counts[action] += 1
        item_action_counts[aid][action] += 1
        first_pos.setdefault(aid, position)
        last_pos[aid] = position
        item_last_action[aid] = action
        item_last_event_ts[aid] = ts
        events_last_hour += int(hour_start <= ts <= prediction_ts)
        events_last_day += int(day_start <= ts <= prediction_ts)

        if action == 'clicks':
            item_last_click_ts[aid] = ts
        elif action == 'carts':
            item_last_cart_ts[aid] = ts
            item_last_cart_pos[aid] = position
        elif action == 'orders':
            item_last_order_pos[aid] = position

    clicks_last_5 = Counter(
        aid for aid, action in zip(session_aids[-5:], session_types[-5:])
        if action == 'clicks'
    )

    session_features = {
        'session_len': session_len,
        'session_unique': unique_items,
        'repeat_ratio': 1.0 - unique_items / session_len if session_len else 0.0,
        'has_history': int(session_len > 0),
        'history_click_count': history_counts['clicks'],
        'history_cart_count': history_counts['carts'],
        'history_order_count': history_counts['orders'],
        'history_cart_share': history_counts['carts'] / max(session_len, 1),
        'history_order_share': history_counts['orders'] / max(session_len, 1),
        'history_last_action': session_types[-1] if session_len else 'none',
        'history_unique_cart_items': len(item_last_cart_pos),
        'history_duration_seconds': (
            (session_ts[-1] - session_ts[0]) / 1000.0 if session_len else 0.0
        ),
        'last_event_gap_seconds': (
            (session_ts[-1] - session_ts[-2]) / 1000.0
            if session_len >= 2 else -1.0
        ),
        'history_events_last_hour': events_last_hour,
        'history_events_last_day': events_last_day,
    }

    session_vector = None
    if session_aids:
        min_ts = session_ts[0]
        max_ts = session_ts[-1]
        time_span = max(max_ts - min_ts, 1.0)
        history_vectors = []
        history_weights = []

        for aid, action, ts in zip(session_aids, session_types, session_ts):
            if aid in model.wv.key_to_index:
                time_weight = (ts - min_ts) / time_span
                history_vectors.append(model.wv[aid])
                history_weights.append(
                    TYPE_WEIGHTS.get(action, 1.0) * time_weight
                )

        if history_vectors:
            history_weights = np.asarray(history_weights, dtype=np.float32)
            weight_sum = history_weights.sum()
            if weight_sum > 0:
                history_weights /= weight_sum
            else:
                history_weights[:] = 1.0 / len(history_weights)
            session_vector = np.average(
                history_vectors, axis=0, weights=history_weights
            )
            norm = np.linalg.norm(session_vector)
            if norm > 0:
                session_vector /= norm

    valid_candidates = [
        aid for aid in candidates if aid in model.wv.key_to_index
    ]
    candidate_vectors = [model.wv[aid] for aid in valid_candidates]
    session_similarities = {}
    last_item_similarities = {}

    if candidate_vectors:
        candidate_matrix = np.asarray(candidate_vectors, dtype=np.float32)
        norms = np.linalg.norm(candidate_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        candidate_matrix /= norms

        if session_vector is not None:
            similarities = candidate_matrix @ session_vector
            session_similarities = dict(zip(
                valid_candidates, similarities.astype(float)
            ))

        if session_aids:
            last_aid = session_aids[-1]
            if last_aid in model.wv.key_to_index:
                last_vector = model.wv[last_aid]
                last_norm = np.linalg.norm(last_vector)
                if last_norm > 0:
                    similarities = candidate_matrix @ (last_vector / last_norm)
                    last_item_similarities = dict(zip(
                        valid_candidates, similarities.astype(float)
                    ))

    pop_list = popular_items.get(event_type, [])
    pop_rank = {aid: rank for rank, aid in enumerate(pop_list)}
    candidate_rank = candidate_rank_map if candidate_rank_map is not None else {
        aid: rank for rank, aid in enumerate(candidates)
    }

    def seconds_since(last_ts):
        return (prediction_ts - last_ts) / 1000.0 if last_ts is not None else -1.0

    rows = []
    for aid in candidates:
        action_counts = item_action_counts.get(aid, {})
        click_count = action_counts.get('clicks', 0)
        cart_count = action_counts.get('carts', 0)
        order_count = action_counts.get('orders', 0)
        rank = candidate_rank[aid]
        popularity_rank = pop_rank.get(aid, 999)
        row = {
            'candidate_rank': rank,
            'candidate_rank_norm': rank / max(CANDIDATE_K - 1, 1),
            'in_session': int(aid in last_pos),
            'count_in_session': click_count + cart_count + order_count,
            'first_pos': first_pos.get(aid, -1),
            'last_pos': last_pos.get(aid, -1),
            'was_clicked': int(click_count > 0),
            'was_carted': int(cart_count > 0),
            'was_ordered': int(order_count > 0),
            'type_count': len(action_counts),
            'item_click_count': click_count,
            'item_cart_count': cart_count,
            'item_order_count': order_count,
            'item_last_action': item_last_action.get(aid, 'none'),
            'item_click_share': click_count / max(history_counts['clicks'], 1),
            'item_click_count_last_5': clicks_last_5.get(aid, 0),
            'item_cart_after_last_order': int(
                item_last_cart_pos.get(aid, -1) > item_last_order_pos.get(aid, -1)
            ),
            'seconds_since_item_last_event': seconds_since(item_last_event_ts.get(aid)),
            'seconds_since_item_last_click': seconds_since(item_last_click_ts.get(aid)),
            'seconds_since_item_last_cart': seconds_since(item_last_cart_ts.get(aid)),
            'pop_rank': popularity_rank,
            'pop_score': (
                1.0 - popularity_rank / max(len(pop_list), 1)
                if aid in pop_rank else 0.0
            ),
            'in_top_10': int(popularity_rank < 10),
            'in_top_50': int(popularity_rank < 50),
            'pos_normalized': (
                last_pos[aid] / max(session_len - 1, 1)
                if aid in last_pos else -1.0
            ),
            'is_last_item': int(aid in last_pos and last_pos[aid] == session_len - 1),
            'is_first_item': int(aid in first_pos and first_pos[aid] == 0),
            'distance_from_end': session_len - 1 - last_pos[aid] if aid in last_pos else -1,
            'i2v_session_sim': session_similarities.get(aid, 0.0),
            'i2v_last_sim': last_item_similarities.get(aid, 0.0),
            'event_type': event_type,
        }
        row.update(session_features)
        row.update(candidate_metadata[aid])
        rows.append(row)
    return rows


def downcast_frame(frame):
    for column in frame.select_dtypes(include=['float', 'float64']).columns:
        frame[column] = frame[column].astype(np.float32)
    for column in frame.select_dtypes(include=['integer']).columns:
        frame[column] = frame[column].astype(np.int32)
    return frame


class Item2VecCatboostRecommender:
    """Хранит CatBoost и только одну активную Item2Vec-модель с 
    индексом в оперативной памяти. Модели Item2Vec меняются в 
    инференсе в зависимости от типа события"""

    def __init__(self, models_dir, catboost_dir, thread_count=4):
        self.models_dir = Path(models_dir)
        self.catboost_dir = Path(catboost_dir)
        self.thread_count = thread_count
        faiss.omp_set_num_threads(thread_count)

        self.ranker = cb.CatBoostClassifier()
        self.ranker.load_model(str(self.catboost_dir / 'catboost_ranker.cbm'))
        with open(self.catboost_dir / 'feature_cols.json') as file:
            self.feature_cols = json.load(file)
        with open(self.catboost_dir / 'categorical_features.json') as file:
            self.cat_features = json.load(file)
        with open(self.catboost_dir / 'popular_items.json') as file:
            self.popular_items = json.load(file)

        self._event_type = None
        self._model = None
        self._index = None
        self._idx_to_aid = None
        self._empty_history_predictions = {}

    def unload_item2vec(self):
        self._event_type = None
        self._model = None
        self._index = None
        self._idx_to_aid = None
        gc.collect()

    def load_event_type(self, event_type):
        if event_type == self._event_type:
            return
        self.unload_item2vec()
        self._model, self._index, self._idx_to_aid = load_model_and_build_index(
            self.models_dir / f'item2vec_{event_type}.model'
        )
        self._event_type = event_type

    def predict_batch(self, sessions, event_type):
        """Возвращает top-20 для каждой сессии в исходном порядке."""
        if not sessions:
            return []
        self.load_event_type(event_type)
        predictions = [None] * len(sessions)
        rows = []
        groups = []

        for position, session in enumerate(sessions):
            events = session['events']
            if not events and event_type in self._empty_history_predictions:
                predictions[position] = list(
                    self._empty_history_predictions[event_type]
                )
                continue

            aids = [str(event['aid']) for event in events]
            actions = [event['type'] for event in events]
            timestamps = [float(event['ts']) for event in events]
            candidates, metadata = generate_candidates(
                aids, actions, timestamps, event_type,
                self._model, self._index, self._idx_to_aid,
                popular_items=self.popular_items,
                type_weights=TYPE_WEIGHTS,
                top_k=CANDIDATE_K,
                last_n=LAST_N,
                return_metadata=True,
            )
            start = len(rows)
            rows.extend(compute_features_for_session(
                aids, actions, timestamps, candidates, event_type, self._model,
                prediction_ts=timestamps[-1] if timestamps else 0.0,
                candidate_metadata=metadata,
                popular_items=self.popular_items,
            ))
            groups.append((position, candidates, start, len(rows), not events))

        if rows:
            frame = downcast_frame(pd.DataFrame.from_records(rows))
            pool = cb.Pool(
                frame[self.feature_cols], cat_features=self.cat_features
            )
            scores = self.ranker.predict_proba(
                pool, thread_count=self.thread_count
            )[:, 1]
        else:
            scores = np.empty(0, dtype=np.float64)

        for position, candidates, start, stop, empty_history in groups:
            order = np.argsort(-scores[start:stop], kind='stable')[:TOP_K]
            ranked = [candidates[int(index)] for index in order]
            predictions[position] = ranked
            if empty_history:
                self._empty_history_predictions[event_type] = tuple(ranked)
        return predictions

    def recommend(self, events):
        """Рекомендации трёх типов для одной истории; модели загружаются по очереди."""
        session = {'events': events}
        return {
            event_type: self.predict_batch([session], event_type)[0]
            for event_type in EVENT_TYPES
        }

Writing /kaggle/working/recommender.py
